# scVI Pretraining — Reichart 2022 Cardiac Atlas

Run this on Google Colab with a T4 GPU (Runtime → Change runtime type → T4 GPU).

Upload `cardiomyocytes.h5ad` to your Google Drive before starting.

**Expected total time: ~45 min**

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU — change runtime type first')

In [ ]:
%pip install -q scvi-tools scanpy umap-learn matplotlib imbalanced-learn xgboost shap
print('Done')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, anndata as ad, numpy as np, scanpy as sc, warnings
warnings.filterwarnings('ignore')

H5AD_PATH = '/content/drive/MyDrive/cardiomyocytes.h5ad'  # CHANGE IF NEEDED
DCM = 'dilated cardiomyopathy'
ACM = 'arrhythmogenic right ventricular cardiomyopathy'

print('Loading...')
# backed='r' avoids loading the full 880k-cell matrix into RAM up front;
# .to_memory() then pulls in only the DCM+ACM rows we actually need (~1.5GB)
adata = ad.read_h5ad(H5AD_PATH, backed='r')
mask = adata.obs['disease'].isin([DCM, ACM])
adata = adata[mask].to_memory()
print(f'Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
print(adata.obs['disease'].value_counts().to_string())

In [ ]:
# Select 3000 HVGs via sparse Fano factor (var/mean) — cheaper than
# scanpy's seurat_v3 flavor, which spikes RAM on Colab's free tier.
# Same method already validated on this dataset in the laptop pipeline.
import scipy.sparse as sp

X = adata.X
if not sp.issparse(X):
    X = sp.csr_matrix(X)
X = X.astype(np.float32)

means = np.asarray(X.mean(axis=0)).ravel()
means_sq = np.asarray(X.power(2).mean(axis=0)).ravel()
variances = means_sq - means**2
fano = variances / np.clip(means, 1e-10, None)

top_idx = np.argsort(fano)[-3000:]
adata_hvg = adata[:, top_idx].copy()
print(f'HVG matrix: {adata_hvg.n_obs:,} x {adata_hvg.n_vars:,}')

In [ ]:
import scvi
scvi.settings.seed = 42

# batch_key='donor_id' = factor out per-patient technical effects
# so the 20 latent numbers capture biology, not patient identity
scvi.model.SCVI.setup_anndata(adata_hvg, batch_key='donor_id')
model = scvi.model.SCVI(adata_hvg, n_latent=20, n_layers=2, n_hidden=128)
print(model)

In [ ]:
# PRETRAINING — no disease labels used
# Model learns: compress 3000 genes -> 20 numbers -> reconstruct 3000 genes
# Watch train_loss drop. Lower = better reconstruction = model is learning.
print('Pretraining on all 880k cells... (~40 min on T4)')
model.train(max_epochs=100, train_size=0.9, early_stopping=True, early_stopping_patience=10)
print('Done!')

In [ ]:
import matplotlib.pyplot as plt
train_hist = model.history['train_loss_epoch']
val_hist   = model.history['validation_loss']
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(train_hist.index, train_hist.values, label='Train loss', color='#3A6BBD', lw=2)
ax.plot(val_hist.index, val_hist.values, label='Val loss', color='#C94040', lw=2, ls='--')
ax.set(xlabel='Epoch', ylabel='ELBO loss (lower = better)', title='Training curve — should drop and plateau')
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig('training_loss.png', dpi=150); plt.show()

In [ ]:
# Extract latent — each cell becomes a 20-number biological fingerprint
# (adata already filtered to DCM+ACM back in cell 3, so no need to re-filter)
print('Extracting latent representations...')
adata_hvg.obsm['X_scVI'] = model.get_latent_representation()

adata_sub = adata_hvg
latent_sub = adata_sub.obsm['X_scVI']
print(f'Latent shape: {latent_sub.shape}  (n_cells, 20)')

In [ ]:
# UMAP — visualise the latent space
# Cell types should cluster, donors should be MIXED (batch correction worked)
print('UMAP... (~5 min)')
sc.pp.neighbors(adata_hvg, use_rep='X_scVI', n_neighbors=15)
sc.tl.umap(adata_hvg)
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sc.pl.umap(adata_hvg, color='disease', ax=axes[0], show=False, title='Disease label')
sc.pl.umap(adata_hvg, color='donor_id', ax=axes[1], show=False, title='Donor (should be mixed — not clustered)')
plt.tight_layout(); plt.savefig('scvi_umap.png', dpi=150); plt.show()

In [ ]:
# HONEST EVALUATION on 20-dim latent
# Same StratifiedGroupKFold as the Phase A pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb

X = latent_sub
y = (adata_sub.obs['disease'] == DCM).astype(int).values
groups = adata_sub.obs['donor_id'].astype('category').cat.codes.values
scale  = y.sum() / (1 - y).sum()

results = {}
for name, clf in [
    ('RF latent',  RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('XGB latent', xgb.XGBClassifier(n_estimators=200, scale_pos_weight=scale, random_state=42, eval_metric='logloss', verbosity=0))
]:
    aucs = []
    for fold, (tr, val) in enumerate(StratifiedGroupKFold(n_splits=5).split(X, y, groups)):
        clf.fit(X[tr], y[tr])
        auc = roc_auc_score(y[val], clf.predict_proba(X[val])[:,1])
        aucs.append(auc)
        print(f'{name} fold {fold+1}: {auc:.4f}')
    results[name] = aucs
    print(f'  Mean: {np.mean(aucs):.4f}\n')

print('=== COMPARISON ===')
print(f'Baseline  RF (raw genes):  0.705')
print(f'Baseline XGB (raw genes):  0.701')
print(f'scVI      RF:  {np.mean(results["RF latent"]):.4f}')
print(f'scVI     XGB:  {np.mean(results["XGB latent"]):.4f}')

In [ ]:
# Summary figure
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('scVI Pretraining vs Raw Gene Baseline', fontsize=13, fontweight='bold')

folds = [1,2,3,4,5]
raw_rf  = [0.6223,0.5659,0.8174,0.6609,0.8583]
raw_xgb = [0.6700,0.5920,0.8099,0.5666,0.8654]

ax = axes[0]
ax.plot(folds, raw_rf,  'o--', color='#C94040', alpha=0.4, label='RF raw genes')
ax.plot(folds, raw_xgb, 's--', color='#3A6BBD', alpha=0.4, label='XGB raw genes')
ax.plot(folds, results['RF latent'],  'o-', color='#C94040', lw=2.5, ms=9, label='RF scVI latent')
ax.plot(folds, results['XGB latent'], 's-', color='#3A6BBD', lw=2.5, ms=9, label='XGB scVI latent')
ax.axhline(0.5, color='gray', ls=':', lw=1, label='Chance')
ax.set(xlabel='Fold', ylabel='Honest AUC', title='Per-fold AUC')
ax.legend(frameon=False, fontsize=9); ax.spines[['top','right']].set_visible(False)

ax = axes[1]
names_b = ['RF\nraw','XGB\nraw','RF\nscVI','XGB\nscVI']
vals_b  = [0.705, 0.701, np.mean(results['RF latent']), np.mean(results['XGB latent'])]
bars = ax.bar(names_b, vals_b, color=['#C94040','#3A6BBD','#C94040','#3A6BBD'], edgecolor='white')
for i in range(2): bars[i].set_alpha(0.35)
for bar, val in zip(bars, vals_b):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.3f}', ha='center', fontweight='bold')
ax.set(ylim=(0.4,0.95), title='Mean AUC'); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig('scvi_result_summary.png', dpi=150); plt.show()

In [ ]:
# Save model + latent to Drive for later use
import os
SAVE_DIR = '/content/drive/MyDrive/scvi_cardiac_model'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save(SAVE_DIR, overwrite=True)
np.save(f'{SAVE_DIR}/latent_dcm_acm.npy', latent_sub)
adata_sub.obs.to_csv(f'{SAVE_DIR}/latent_obs.csv')
print(f'Saved to {SAVE_DIR}')